In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import os, sys
import warnings
warnings.filterwarnings('ignore')

BASE = r"C:\Users\Admin\OneDrive\Desktop\AI-Business-Risk-Intelligence"
sys.path.append(BASE)

df = pd.read_csv(os.path.join(
    BASE, "data", "processed",
    "telco_with_predictions.csv"))

print("✅ Data loaded!")
print(f"Total customers: {len(df)}")

✅ Data loaded!
Total customers: 7043


In [3]:
from src.models.ltv_decay_predictor import (
    calculate_customer_ltv,
    calculate_intervention_roi
)

# Test highest risk customer
idx = df['churn_prob_30day'].idxmax()
row = df.iloc[idx]

monthly = float(row.get('MonthlyCharges', 65))
tenure  = float(row.get('tenure', 12))
c30     = float(row['churn_prob_30day'])
c60     = float(row['churn_prob_60day'])
c90     = float(row['churn_prob_90day'])

ltv = calculate_customer_ltv(
    monthly, tenure, c30, c60, c90)

print(f"Customer {idx} — LTV Decay Analysis")
print(f"{'='*45}")
print(f"Current LTV      : ₹{ltv['current_ltv']:,.0f}/year")
print(f"Value at Risk    : ₹{ltv['value_at_risk']:,.0f}")
print(f"Decay Rate       : {ltv['decay_rate_pct']:.1f}% in 3 months")
print(f"Decay Pattern    : {ltv['decay_pattern']}")
print(f"Turning Point    : Month {ltv['turning_point']}")
print(f"Best Action Month: Month {ltv['intervention_month']}")
print(f"\nMonth by Month:")
for m, v in zip(ltv['months'], ltv['decay_curve']):
    bar = "█" * int(v / ltv['current_ltv'] * 20)
    print(f"  Month {m:2d}: ₹{v:6.0f} {bar}")

Customer 4453 — LTV Decay Analysis
Current LTV      : ₹1,182/year
Value at Risk    : ₹1,182
Decay Rate       : 91.4% in 3 months
Decay Pattern    : 🔴 CRITICAL DECAY
Turning Point    : Month 1
Best Action Month: Month 1

Month by Month:
  Month  0: ₹  1182 ████████████████████
  Month  1: ₹    15 
  Month  2: ₹    47 
  Month  3: ₹   102 █
  Month  4: ₹     0 
  Month  5: ₹     0 
  Month  6: ₹     0 
  Month  7: ₹     0 
  Month  8: ₹     0 
  Month  9: ₹     0 
  Month 10: ₹     0 
  Month 11: ₹     0 
  Month 12: ₹     0 


In [ ]:
roi = calculate_intervention_roi(
    ltv, 'discount_20', monthly)

print(f"\n💰 INTERVENTION ROI ANALYSIS")
print(f"{'='*45}")
print(f"Action           : 20% Discount")
print(f"Action Cost      : ₹{roi['action_cost']:,.0f}")
print(f"Value Saved      : ₹{roi['value_saved']:,.0f}")
print(f"Net ROI          : ₹{roi['net_roi']:,.0f}")
print(f"ROI %            : {roi['roi_percentage']:.0f}%")
print(f"Verdict          : {roi['verdict']}")
print(f"Best Month to Act: Month {roi['best_month_to_act']}")